In [1]:
import json
import time
import datetime
import requests
import pandas as pd
import os
import re
import matplotlib.pyplot as plt
from datetime import datetime


## Load country regrex

In [2]:
# Confirm current working directory
cwd = os.getcwd()
print(f"Current working directory: {cwd}")

Current working directory: C:\Users\wb663259\OneDrive - WBG\Desktop\Ulrich\3. Uncertainty\code


In [3]:
regrex_file = f"{cwd}/query_regrex_all.csv"

In [4]:
# Assumes 'query_regrex.csv' has columns: ISO3, regex
df_regex = pd.read_csv(regrex_file, dtype=str, encoding="latin1")

# Build a dict:  country_info["USA"]  ->  {"regex":   r"(\b)(united...)", 
#                                          "region": "usa"}
country_info = (
    df_regex
    .set_index('iso3')[['regex', 'region_code']]
    .to_dict('index')
)

## Query templates

In [5]:
def inject_country(template: str, regex: str, region_code: str) -> str:
    return (template
            .replace('__COUNTRY_REGEX__', regex)
            .replace('__REGION_CODE__',  region_code.lower()))  # region-codes are stored lowercase

In [6]:
api_key = ''
key_headers = {
    'user-key': 'CyYzoIStUsa5wgdSTeSOglpPU4gaFxJV', # Enter your API key here
    'content-type': "application/json",
    'cache-control': "no-cache",
    'X-API-VERSION' : "2.0"
}

In [7]:
# ============================================================
# FPU SUB-CATEGORY KEYWORDS
# Aggregate extraction rule: Tax OR Expenditure OR Debt OR Residual
# Reported sub-indices: Tax, Expenditure, Debt (Residual is NOT reported).
# Article labels are non-mutually-exclusive (multi-label).
# Token lists follow variant C1b of 02. Revised keywords/subindex_clauses.py.
# ============================================================

TAX_PATTERN = (
    r"tax"
    r"|taxed"
    r"|taxation"
    r"|taxes"
)

EXPENDITURE_PATTERN = (
    r"government\W+spending"
    r"|federal\W+spending"
    r"|public\W+spending"
    r"|government\W+expenditure\w*"
    r"|federal\W+expenditure\w*"
    r"|public\W+expenditure\w*"
    r"|defense\W+spending"
    r"|defence\W+spending"
    r"|military\W+spending"
    r"|pension\W+reform\w*"
    r"|pension\W+expenditure\w*"
    r"|healthcare\W+expenditure\w*"
    r"|medical\W+care\W+expenditure\w*"
    r"|social\W+expenditure\w*"
    r"|social\W+safety\W+net\w*"
    r"|public\W+investment\w*"
    r"|subsidy"
    r"|subsidies"
    r"|subsidi[sz]\w*"
    r"|entitlement\W+spending"
    r"|social\W+security"
    r"|fiscal\W+stimulus"
    r"|social\W+protection"
    r"|social\W+securit\w*\W+expenditure\w*"
    r"|defense\W+expenditure\w*"
    r"|defence\W+expenditure\w*"
    r"|military\W+expenditure\w*"
    r"|pension\W+spending"
    r"|health\W+care\W+expenditure\w*"
    r"|health\W+spending"
    r"|health\W*care\W+spending"
    r"|medical\W+care\W+spending"
    r"|social\W+spending"
    r"|fiscal\W+cliff"
)

DEBT_PATTERN = (
    r"federal\W+debt"
    r"|government\W+debt"
    r"|national\W+debt"
    r"|public\W+debt"
    r"|sovereign\W+debt"
    r"|debt\W+burden\w*"
    r"|debt\W+repay\w*"
    r"|debt\W+sustainability"
    r"|government\W+borrowing"
    r"|sovereign\W+borrowing"
    r"|public\W+borrowing"
    r"|national\W+borrowing"
    r"|debt\W+consolidation"
    r"|sovereign\W+default\w*"
    r"|debt\W+default\w*"
    r"|debt\W+restructur\w*"
    r"|foreign\W+debt"
    r"|external\W+debt"
    r"|government\W+bond\w*"
    r"|sovereign\W+bond\w*"
    r"|sovereign\W+yield\w*"
    r"|balanced\W+budget"
    r"|balance\W+the\W+budget"
    r"|budget\W+deficit\w*"
    r"|government\W+deficit\w*"
    r"|national\W+deficit\w*"
    r"|federal\W+deficit\w*"
    r"|budget\W+gap\w*"
    r"|debt\W+ceiling\w*"
    r"|sovereign\W+risk\w*"
    r"|fiscal\W+deficit\w*"
)

# Residual fiscal-topic terms. These are part of the AGGREGATE FPU numerator
# (so that FPU_base remains comparable with the published baseline FPU index)
# but are NOT reported as a sub-index.
RESIDUAL_PATTERN = (
    r"fiscal\W+polic\w*"
    r"|public\W+finance\w*"
    r"|public\W+sector"
    r"|government\W+budget\w*"
    r"|national\W+budget\w*"
    r"|supplementary\W+budget\w*"
)

# IMPORTANT: OR across the three reported categories PLUS the residual bucket.
# This is Clause C of the baseline FPU query, partitioned into four buckets.
FISCAL_PATTERN = rf"(?:{TAX_PATTERN}|{EXPENDITURE_PATTERN}|{DEBT_PATTERN}|{RESIDUAL_PATTERN})"

POLICY_PATTERN = (
    r"legislat\w*|minister|policies|policy|regulat\w*|government|"
    r"congress\w*|parliament|national\W+assembly|ministry"
)

UNCERTAINTY_PATTERN = (
    r"doubt|doubtful|risk\w*|unresolved|unsettled|dubious|undetermined|"
    r"undecided|unpredictable|precarious|ambiguous|uncertain\w*|volatilit\w*"
)

ECONOMIC_PATTERN = r"economic|economy"

# Reverted verbatim to the published baseline pipeline
# (05. Replication/Factiva-FPU/01. FPU_Factiva_all.ipynb, numerator_rece_where).
# Literal single spaces are intentional -- do NOT replace them with \W+ ; doing so
# would silently break continuity with the published recession-excluded FPU series.
RECESSION_PATTERN = (
    r"recession\w{0,}|"
    r"econom\w{0,}\W+(?:\w+\W+){0,5}?(downturn\w{0,}|slowdown\w{0,}|slow\w{0,} down|contract\w{0,}|slump\w{0,}|collapse\w{0,}|stagnat\w{0,}|depress\w{0,}|meltdown)|"
    r"(downturn\w{0,}|slowdown\w{0,}|slow\w{0,} down|contract\w{0,}|slump\w{0,}|stagnat\w{0,}|depress\w{0,}|meltdown|collapse\w{0,})\W+(?:\w+\W+){0,5}?econom\w{0,}|"
    r"GDP\W+(?:\w+\W+){0,5}?(contract\w{0,}|declin\w{0,}|shrink\w{0,})|"
    r"(contract\w{0,}|declin\w{0,}|shrink\w{0,})\W+(?:\w+\W+){0,5}?GDP|"
    r"negative GDP growth|"
    r"(output|manufacturing)\W+(?:\w+\W+){0,5}?(contract\w{0,}|fall\w{0,}|slump\w{0,})|"
    r"(contract\w{0,}|fall\w{0,}|slump\w{0,})\W+(?:\w+\W+){0,5}?(output|manufacturing)|"
    r"financial crisis|economic crisis|banking crisis|credit crisis|"
    r"currency crisis|credit crunch|liquidity crisis|output collapse|market collapse"
)

SOURCE_PATTERN = (
    r"(^|,)(ctbonl|sfc|j|dal|usatonl|wp|gma|absr|wnsa|wnt|nlne|thwk|wnsu|"
    r"abcaze|bbcmna|bbcmre|bbcrnd|bbcpsm|sxmn|cbst|cbsa|fcnt|ntln|toda|t|"
    r"grdn|telem|lba|glob|ec|stel|bbcap|bbcmap|bbcapp|bbccau|bbcca|bbceup|"
    r"bbcsup|bbcmm|bbcmep|bbcmnf|bbcsap|bbcukb|aprs)($|,)"
)

TEXT_EXPR = "CONCAT(title, ' ', IFNULL(snippet, ''), ' ', IFNULL(body, ''), ' ', IFNULL(section, ''))"

In [8]:
# ============================================================
# FPU QUERY BUILDERS
# Aggregate extraction rule: Tax OR Expenditure OR Debt OR Residual
# Subcategory queries use the same common FPU conditions and differ
# only in the fiscal-category pattern.
# Each rule is built twice: with and without the recession exclusion.
# ============================================================
# NOTE: the rf"" prefixes below are required. With a plain f"" literal, Python
# interprets \b as the backspace character (0x08) instead of passing the two
# characters \ and b through to the BigQuery regex, which silently voids every
# word-boundary anchor in the query.

def build_fpu_where(fiscal_pattern: str, exclude_recession: bool = False) -> str:
    where = (
        rf"REGEXP_CONTAINS({TEXT_EXPR}, r'(?i)(\b)(?:{POLICY_PATTERN})(\b)')"
        rf" AND REGEXP_CONTAINS({TEXT_EXPR}, r'(?i)(\b)(?:{UNCERTAINTY_PATTERN})(\b)')"
        rf" AND REGEXP_CONTAINS({TEXT_EXPR}, r'(?i)(\b)(?:{fiscal_pattern})(\b)')"
        rf" AND REGEXP_CONTAINS({TEXT_EXPR}, r'(?i)(\b)(?:{ECONOMIC_PATTERN})(\b)')"
        r" AND REGEXP_CONTAINS(CONCAT(title, ' ', IFNULL(snippet, '')), r'(?i)__COUNTRY_REGEX__')"
        rf" AND REGEXP_CONTAINS(restrictor_codes, r'(?i){SOURCE_PATTERN}')"
        r" AND LOWER(region_codes) LIKE '%,__REGION_CODE__,%'"
    )
    if exclude_recession:
        where += rf" AND NOT REGEXP_CONTAINS({TEXT_EXPR}, r'(?i)(\b)(?:{RECESSION_PATTERN})(\b)')"
    return where


def build_other_where(exclude_recession: bool = False) -> str:
    """
    Count aggregate-FPU articles that fall in the residual fiscal bucket
    but do NOT match Tax, Expenditure, or Debt.

    This is the explicit Other fiscal category:
        Residual AND NOT(Tax OR Expenditure OR Debt)

    Existing aggregate/Tax/Expenditure/Debt query definitions are left unchanged,
    so their existing cache payload hashes remain unchanged.
    """
    where = build_fpu_where(RESIDUAL_PATTERN, exclude_recession=exclude_recession)
    where += (
        rf" AND NOT REGEXP_CONTAINS({TEXT_EXPR}, "
        rf"r'(?i)(\b)(?:{TAX_PATTERN}|{EXPENDITURE_PATTERN}|{DEBT_PATTERN})(\b)')"
    )
    return where


# Aggregate numerator: ANY of the four fiscal buckets (OR)
numerator_base_where = build_fpu_where(FISCAL_PATTERN, exclude_recession=False)

# Recession-excluded aggregate numerator: same OR rule
numerator_rece_where = build_fpu_where(FISCAL_PATTERN, exclude_recession=True)

# Category-specific numerators. These are separate count queries used to
# construct Tax / Expenditure / Debt FPU while allowing overlap across categories.
tax_where = build_fpu_where(TAX_PATTERN, exclude_recession=False)
expenditure_where = build_fpu_where(EXPENDITURE_PATTERN, exclude_recession=False)
debt_where = build_fpu_where(DEBT_PATTERN, exclude_recession=False)

# Recession-excluded category numerators. Same three category rules, with the
# recession filter applied, so each sub-index has a recession-excluded twin.
tax_rece_where = build_fpu_where(TAX_PATTERN, exclude_recession=True)
expenditure_rece_where = build_fpu_where(EXPENDITURE_PATTERN, exclude_recession=True)
debt_rece_where = build_fpu_where(DEBT_PATTERN, exclude_recession=True)

# Explicit Other fiscal numerators.
# These are the only NEW Factiva query definitions added by this revision.
other_where = build_other_where(exclude_recession=False)
other_rece_where = build_other_where(exclude_recession=True)

# Sanity check: the backspace character must never appear in a where clause.
for _name, _w in [
    ("numerator_base", numerator_base_where), ("numerator_rece", numerator_rece_where),
    ("tax", tax_where), ("tax_rece", tax_rece_where),
    ("expenditure", expenditure_where), ("expenditure_rece", expenditure_rece_where),
    ("debt", debt_where), ("debt_rece", debt_rece_where),
    ("other", other_where), ("other_rece", other_rece_where),
]:
    assert "\x08" not in _w, f"{_name}_where contains a literal backspace: use rf'' not f''"
print("Built 10 numerator where-clauses (8 existing + 2 Other); word-boundary escaping verified.")

Built 8 numerator where-clauses; word-boundary escaping verified.


In [9]:
# Query logic check
#
# Aggregate FPU article count:
#   Policy AND Uncertainty AND (Tax OR Expenditure OR Debt OR Residual) AND Economy ...
#
# Subcategory counts:
#   Policy AND Uncertainty AND Tax AND Economy ...
#   Policy AND Uncertainty AND Expenditure AND Economy ...
#   Policy AND Uncertainty AND Debt AND Economy ...
#   Policy AND Uncertainty AND Residual AND NOT(Tax OR Expenditure OR Debt) AND Economy ... [Other]
#
# Each rule above is built twice: once as-is, and once with the recession
# exclusion added (AND NOT Recession), giving 11 count series per country-date:
#   numerator_base_count, numerator_rece_count,
#   tax_count, tax_rece_count,
#   expenditure_count, expenditure_rece_count,
#   debt_count, debt_rece_count,
#   other_count, other_rece_count,
#   denominator_count
#
# A single article may therefore be counted in more than one subcategory.
# Do NOT calculate aggregate numerator as tax_count + expenditure_count + debt_count
# (subcategory overlap prevents arithmetic recovery of the aggregate).
#
# Other is queried directly as Residual AND NOT(Tax OR Expenditure OR Debt);
# it is NOT calculated as aggregate - tax - expenditure - debt.
#
# Each *_rece_count is a subset of BOTH its own plain twin (e.g. tax_rece_count
# <= tax_count) and of the recession-excluded aggregate (e.g. tax_rece_count
# <= numerator_rece_count).
print("Aggregate fiscal rule: Tax OR Expenditure OR Debt OR Residual")
print("Subcategory counts preserve overlap across Tax / Expenditure / Debt.")
print("11 count series per country-date: numerator_base, numerator_rece, tax, tax_rece,")
print("expenditure, expenditure_rece, debt, debt_rece, other, other_rece, denominator.")
print("Sub-category counts overlap and must not be summed to recover the aggregate.")
print("Each *_rece_count is a subset of both its own plain count and of numerator_rece_count.")

Aggregate fiscal rule: Tax OR Expenditure OR Debt OR Residual
Subcategory counts preserve overlap across Tax / Expenditure / Debt.
9 count series per country-date: numerator_base, numerator_rece, tax, tax_rece,
expenditure, expenditure_rece, debt, debt_rece, denominator.
Sub-category counts overlap and must not be summed to recover the aggregate.
Each *_rece_count is a subset of both its own plain count and of numerator_rece_count.


In [10]:
# Denominator is unchanged: all qualifying news for the country/source/region.
denominator_where = (
    "REGEXP_CONTAINS(CONCAT(title, ' ', IFNULL(snippet, '')), r'(?i)__COUNTRY_REGEX__')"
    f" AND REGEXP_CONTAINS(restrictor_codes, r'(?i){SOURCE_PATTERN}')"
    " AND LOWER(region_codes) LIKE '%,__REGION_CODE__,%'"
)


### Article-level Tax / Expenditure / Debt labels

The current Dow Jones Analytics query used below returns aggregated `publication_datetime` + `count` results. Therefore the existing country loop continues to produce the aggregate FPU series exactly as before, but with the revised `(Tax OR Expenditure OR Debt OR Residual)` extraction rule. Each numerator query is also built as a recession-excluded twin (`AND NOT Recession`), so the aggregate and each reported sub-index (Tax, Expenditure, Debt) has a `*_rece` counterpart.

The helper below is for article-level records when `title`, `snippet`, `body`, and/or `section` are available. Labels are **multi-label**, so the same article may have `tax=1` and `debt=1` at the same time. No `not_assigned` category is created.

In [11]:
# ============================================================
# ARTICLE-LEVEL MULTI-LABELING HELPERS
# No not_assigned category.
# NOTE: this helper is NOT used by the Analytics count pipeline above, which
# returns aggregated date+count rows only. It is provided for article-level
# work when full text (title/snippet/body/section) is available.
# ============================================================

TAX_RE = re.compile(rf"(?i)(\b)(?:{TAX_PATTERN})(\b)")
EXPENDITURE_RE = re.compile(rf"(?i)(\b)(?:{EXPENDITURE_PATTERN})(\b)")
DEBT_RE = re.compile(rf"(?i)(\b)(?:{DEBT_PATTERN})(\b)")
RESIDUAL_RE = re.compile(rf"(?i)(\b)(?:{RESIDUAL_PATTERN})(\b)")


def _matched_terms(text, regex):
    if pd.isna(text):
        return []
    return sorted({m.group(0).strip() for m in regex.finditer(str(text))})


def label_fpu_articles(article_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add non-mutually-exclusive Tax / Expenditure / Debt labels.

    Expected article text columns (missing columns are treated as blank):
        title, snippet, body, section

    Output labels:
        tax, expenditure, debt   (0/1)
        tax_keywords, expenditure_keywords, debt_keywords
        n_categories, category_combination
        tax_expenditure, tax_debt, expenditure_debt, all_three
    """
    df = article_df.copy()

    for col in ["title", "snippet", "body", "section"]:
        if col not in df.columns:
            df[col] = ""

    df["article_text"] = (
        df["title"].fillna("").astype(str) + " "
        + df["snippet"].fillna("").astype(str) + " "
        + df["body"].fillna("").astype(str) + " "
        + df["section"].fillna("").astype(str)
    )

    df["tax"] = df["article_text"].str.contains(TAX_RE, na=False).astype("int8")
    df["expenditure"] = df["article_text"].str.contains(EXPENDITURE_RE, na=False).astype("int8")
    df["debt"] = df["article_text"].str.contains(DEBT_RE, na=False).astype("int8")

    df["tax_keywords"] = df["article_text"].apply(lambda x: "; ".join(_matched_terms(x, TAX_RE)))
    df["expenditure_keywords"] = df["article_text"].apply(lambda x: "; ".join(_matched_terms(x, EXPENDITURE_RE)))
    df["debt_keywords"] = df["article_text"].apply(lambda x: "; ".join(_matched_terms(x, DEBT_RE)))

    df["n_categories"] = df[["tax", "expenditure", "debt"]].sum(axis=1).astype("int8")

    def _combination(row):
        labels = []
        if row["tax"] == 1:
            labels.append("Tax")
        if row["expenditure"] == 1:
            labels.append("Expenditure")
        if row["debt"] == 1:
            labels.append("Debt")
        return " + ".join(labels) if labels else "Check"

    df["category_combination"] = df.apply(_combination, axis=1)

    df["tax_expenditure"] = ((df["tax"] == 1) & (df["expenditure"] == 1)).astype("int8")
    df["tax_debt"] = ((df["tax"] == 1) & (df["debt"] == 1)).astype("int8")
    df["expenditure_debt"] = ((df["expenditure"] == 1) & (df["debt"] == 1)).astype("int8")
    df["all_three"] = ((df["tax"] == 1) & (df["expenditure"] == 1) & (df["debt"] == 1)).astype("int8")

    return df

In [12]:
def build_payload(where_clause: str) -> dict:
    """Return the dict to pass to requests.post."""
    return {
        "query": {
            "where": where_clause,
            "top": -1,
            "format": "json"      # JSON easier to parse than CSV; change if you prefer
        }
    }

## Functions to one query

In [13]:
analytics_url = "https://api.dowjones.com/analytics"

In [14]:
def run_factiva_query(payload: dict) -> dict:
    """Submit a query, poll until finished, return the final JSON."""
    # 1) submit
    start = time.time()
    resp = requests.post(analytics_url,
                         data=json.dumps(payload),
                         headers=key_headers,
                         timeout=60)
    resp.raise_for_status()

    job = resp.json()
    job_url   = job["links"]["self"]
    job_state = job["data"]["attributes"]["current_state"]

    # 2) poll
    while job_state not in {"JOB_STATE_DONE", "JOB_STATE_FAILED"}:
        time.sleep(10)
        poll = requests.get(job_url, headers=key_headers, timeout=60)
        poll.raise_for_status()
        job_state = poll.json()["data"]["attributes"]["current_state"]

    # 3) final fetch
    final = requests.get(job_url, headers=key_headers, timeout=60)
    final.raise_for_status()
    # 4) courtesy wait 6 min before returning control
    elapsed = time.time() - start
    wait = max(0, 360 - elapsed)
    if wait:
        print(f"Waiting {int(wait)} s for rate limit …")
        time.sleep(wait)
    return final.json()

## Loop over countries

In [15]:
records = []


def _prepare_count_df(rows, output_count_name):
    """Convert one Factiva Analytics count response to date + named count."""
    df = pd.DataFrame(rows)

    if "publication_datetime" not in df.columns:
        return pd.DataFrame(columns=["publication_datetime", output_count_name])

    df["publication_datetime"] = pd.to_datetime(df["publication_datetime"])

    if "count" not in df.columns:
        df["count"] = 0

    # Guard against duplicate date rows in an API response.
    df = (
        df.groupby("publication_datetime", as_index=False)["count"]
          .sum()
          .rename(columns={"count": output_count_name})
    )
    return df


# Module-level list of the 11 count columns. Reused for the empty-frame
# fallback and the column-fill loop below so the three places can never
# drift apart.
COUNT_COLS = [
    "numerator_base_count", "numerator_rece_count",
    "tax_count", "tax_rece_count",
    "expenditure_count", "expenditure_rece_count",
    "debt_count", "debt_rece_count",
    "other_count", "other_rece_count",
    "denominator_count",
]

print("11 query slots per country; existing 9 are cache-resumed, so only the 2 new Other queries should call Factiva when prior caches are present.")

total_countries = len(country_info)
completed_countries = 0
print(f"Total countries to process: {total_countries}")

for iso, info in country_info.items():
    print(f"=== {iso} ===")

    regex = info['regex']
    region_code = info['region_code']

    # Inject country-specific terms into all query templates.
    where_num_base = inject_country(numerator_base_where, regex, region_code)
    where_num_rece = inject_country(numerator_rece_where, regex, region_code)
    where_tax = inject_country(tax_where, regex, region_code)
    where_tax_rece = inject_country(tax_rece_where, regex, region_code)
    where_expenditure = inject_country(expenditure_where, regex, region_code)
    where_expenditure_rece = inject_country(expenditure_rece_where, regex, region_code)
    where_debt = inject_country(debt_where, regex, region_code)
    where_debt_rece = inject_country(debt_rece_where, regex, region_code)
    where_other = inject_country(other_where, regex, region_code)
    where_other_rece = inject_country(other_rece_where, regex, region_code)
    where_den = inject_country(denominator_where, regex, region_code)

    # Build payloads.
    payloads = {
        "numerator_base_count": build_payload(where_num_base),
        "numerator_rece_count": build_payload(where_num_rece),
        "tax_count": build_payload(where_tax),
        "tax_rece_count": build_payload(where_tax_rece),
        "expenditure_count": build_payload(where_expenditure),
        "expenditure_rece_count": build_payload(where_expenditure_rece),
        "debt_count": build_payload(where_debt),
        "debt_rece_count": build_payload(where_debt_rece),
        "other_count": build_payload(where_other),
        "other_rece_count": build_payload(where_other_rece),
        "denominator_count": build_payload(where_den),
    }

    # Run each count query separately. This preserves multi-category overlap:
    # one article may contribute to tax_count and debt_count simultaneously.
    #
    # RESUME/CACHE LOGIC
    # ------------------
    # Each successful country-query result is saved immediately. If execution
    # stops later (network/DNS/rate-limit/etc.), the next run reloads completed
    # queries from cache and calls Factiva only for the first unfinished query.
    # The payload hash is part of the filename so changed query definitions do
    # not accidentally reuse stale cached results.
    import hashlib

    cache_dir = os.path.join(cwd, "outputs", "factiva_query_cache")
    os.makedirs(cache_dir, exist_ok=True)

    dfs = []
    for count_name, payload in payloads.items():
        payload_text = json.dumps(payload, sort_keys=True, ensure_ascii=False)
        payload_hash = hashlib.sha256(payload_text.encode("utf-8")).hexdigest()[:12]
        cache_file = os.path.join(
            cache_dir,
            f"{iso}__{count_name}__{payload_hash}.csv"
        )

        if os.path.exists(cache_file):
            print(f"  -> {count_name}: CACHE FOUND — skipping API call")
            df_piece = pd.read_csv(cache_file)
            if "publication_datetime" in df_piece.columns:
                df_piece["publication_datetime"] = pd.to_datetime(
                    df_piece["publication_datetime"]
                )
            dfs.append(df_piece)
            continue

        print(f"  -> {count_name}: querying Factiva")

        # Deliberately let an exception stop the notebook. Completed queries
        # above are already cached, so rerunning resumes at this unfinished one.
        result_json = run_factiva_query(payload)
        rows = result_json["data"]["attributes"].get("results", [])
        df_piece = _prepare_count_df(rows, count_name)

        # Save only AFTER the API call and preparation completed successfully.
        df_piece.to_csv(cache_file, index=False)
        print(f"     STAGE 1 CSV CACHE SAVED: {cache_file}")
        dfs.append(df_piece)

    # Outer-merge all date-level counts.
    merged = None
    for df_piece in dfs:
        if merged is None:
            merged = df_piece.copy()
        else:
            merged = pd.merge(
                merged,
                df_piece,
                on="publication_datetime",
                how="outer"
            )

    if merged is None or merged.empty:
        merged = pd.DataFrame(columns=["publication_datetime"] + COUNT_COLS)

    for col in COUNT_COLS:
        if col not in merged.columns:
            merged[col] = 0

    merged[COUNT_COLS] = merged[COUNT_COLS].fillna(0)
    merged["iso3"] = iso
    # ========================================================
    # STAGE 2 — COUNTRY-LEVEL EXCEL CHECKPOINT
    # ========================================================
    # Once all 11 query slots for this country are available (from API or cache),
    # save a human-readable workbook immediately. This is separate from the
    # CSV cache used for resume and from the final integrated master workbook.
    country_excel_dir = os.path.join(cwd, "outputs", "country_excel")
    os.makedirs(country_excel_dir, exist_ok=True)
    country_excel_file = os.path.join(
        country_excel_dir,
        f"{iso}_Factiva.xlsx"
    )

    with pd.ExcelWriter(country_excel_file, engine="openpyxl") as writer:
        # Main country-level merged date/count table.
        merged.sort_values("publication_datetime").to_excel(
            writer, index=False, sheet_name="merged_counts"
        )

        # Also preserve each of the 11 query outputs as its own sheet.
        # Excel sheet names are limited to 31 characters.
        for count_name, df_piece in zip(payloads.keys(), dfs):
            sheet_name = count_name[:31]
            if "publication_datetime" in df_piece.columns:
                df_to_save = df_piece.sort_values("publication_datetime")
            else:
                df_to_save = df_piece
            df_to_save.to_excel(
                writer, index=False, sheet_name=sheet_name
            )

    print(f"  COUNTRY EXCEL SAVED: {country_excel_file}")

    completed_countries += 1
    pct_complete = (completed_countries / total_countries * 100) if total_countries else 100.0
    print(
        f"✓ {iso} 완료 → {completed_countries}/{total_countries} countries completed "
        f"({pct_complete:.1f}%)"
    )

    records.append(merged)

9 queries now run per country (was 6); expect roughly 1.5x the wall-clock time per country.


Total countries to process: 116
=== ALB ===
  -> numerator_base_count: CACHE FOUND — skipping API call


  -> numerator_rece_count: CACHE FOUND — skipping API call
  -> tax_count: CACHE FOUND — skipping API call


  -> tax_rece_count: CACHE FOUND — skipping API call
  -> expenditure_count: CACHE FOUND — skipping API call


  -> expenditure_rece_count: CACHE FOUND — skipping API call


...

  COUNTRY EXCEL SAVED: C:\Users\wb663259\OneDrive - WBG\Desktop\Ulrich\3. Uncertainty\code\outputs\country_excel\DNK_Factiva.xlsx
✓ DNK 완료 → 65/116 countries completed (56.0%)
=== DZA ===
  -> numerator_base_count: CACHE FOUND — skipping API call
  -> numerator_rece_count: CACHE FOUND — skipping API call
  -> tax_count: CACHE FOUND — skipping API call
  -> tax_rece_count: CACHE FOUND — skipping API call
  -> expenditure_count: CACHE FOUND — skipping API call
  -> expenditure_rece_count: CACHE FOUND — skipping API call
  -> debt_count: CACHE FOUND — skipping API call
  -> debt_rece_count: CACHE FOUND — skipping API call
  -> denominator_count: CACHE FOUND — skipping API call


  COUNTRY EXCEL SAVED: C:\Users\wb663259\OneDrive - WBG\Desktop\Ulrich\3. Uncertainty\code\outputs\country_excel\DZA_Factiva.xlsx
✓ DZA 완료 → 66/116 countries completed (56.9%)
=== EGY ===
  -> numerator_base_count: CACHE FOUND — skipping API call
  -> numerator_rece_count: CACHE FOUND — skipping API call
  -> tax_count: CACHE FOUND — skipping API call
  -> tax_rece_count: CACHE FOUND — skipping API call
  -> expenditure_count: CACHE FOUND — skipping API call
  -> expenditure_rece_count: CACHE FOUND — skipping API call
  -> debt_count: CACHE FOUND — skipping API call
  -> debt_rece_count: CACHE FOUND — skipping API call
  -> denominator_count: CACHE FOUND — skipping API call


  COUNTRY EXCEL SAVED: C:\Users\wb663259\OneDrive - WBG\Desktop\Ulrich\3. Uncertainty\code\outputs\country_excel\EGY_Factiva.xlsx
✓ EGY 완료 → 67/116 countries completed (57.8%)
=== EST ===
  -> numerator_base_count: CACHE FOUND — skipping API call
  -> numerator_rece_count: CACHE FOUND — skipping API call
  -> tax_count: CACHE FOUND — skipping API call
  -> tax_rece_count: CACHE FOUND — skipping API call
  -> expenditure_count: CACHE FOUND — skipping API call
  -> expenditure_rece_count: CACHE FOUND — skipping API call
  -> debt_count: CACHE FOUND — skipping API call
  -> debt_rece_count: CACHE FOUND — skipping API call
  -> denominator_count: CACHE FOUND — skipping API call


  COUNTRY EXCEL SAVED: C:\Users\wb663259\OneDrive - WBG\Desktop\Ulrich\3. Uncertainty\code\outputs\country_excel\EST_Factiva.xlsx
✓ EST 완료 → 68/116 countries completed (58.6%)
=== KHM ===
  -> numerator_base_count: CACHE FOUND — skipping API call
  -> numerator_rece_count: CACHE FOUND — skipping API call
  -> tax_count: CACHE FOUND — skipping API call
  -> tax_rece_count: querying Factiva


HTTPError: 502 Server Error: Bad Gateway for url: https://api.dowjones.com/analytics

## Final Dataframe

In [ ]:
final_df_raw = pd.concat(records, ignore_index=True)
# Optional: sort
final_df_raw = final_df_raw.sort_values(["iso3", "publication_datetime"])

print("\n=== SAMPLE OUTPUT ===")
print(final_df_raw.head())

In [ ]:
# 1. Ensure publication_datetime is datetime
final_df_raw['publication_datetime'] = pd.to_datetime(final_df_raw['publication_datetime'])

In [ ]:
# Notebook 02 applies the 1995 cut-off. Notebook 01 therefore exports the full
# history by default so the raw archive stays complete. Set this to True only if
# you want notebook 01's monthly/quarterly sheets truncated as well.
TRUNCATE_FROM_1995 = False

if TRUNCATE_FROM_1995:
    final_df = final_df_raw[
        final_df_raw['publication_datetime'] >= pd.Timestamp('1995-01-01')
    ].copy()
else:
    final_df = final_df_raw.copy()

print(f"TRUNCATE_FROM_1995={TRUNCATE_FROM_1995}; final_df rows: {len(final_df):,}")

In [ ]:
# Truncation behaviour is now controlled by TRUNCATE_FROM_1995 in the cell above.

### Quarterly

In [ ]:
# 2) Group to quarters
quarterly_count_cols = COUNT_COLS

quarterly = (
    final_df
    .groupby([
        'iso3',
        pd.Grouper(key='publication_datetime', freq='QE')
    ])[quarterly_count_cols]
    .sum()
    .reset_index()
)

# 3) Raw quarterly indices. The same denominator is used for all components.
quarterly['FPU_base_raw'] = quarterly['numerator_base_count'] / quarterly['denominator_count']
quarterly['FPU_excl_rec_raw'] = quarterly['numerator_rece_count'] / quarterly['denominator_count']
quarterly['Tax_FPU_raw'] = quarterly['tax_count'] / quarterly['denominator_count']
quarterly['Tax_FPU_excl_rec_raw'] = quarterly['tax_rece_count'] / quarterly['denominator_count']
quarterly['Expenditure_FPU_raw'] = quarterly['expenditure_count'] / quarterly['denominator_count']
quarterly['Expenditure_FPU_excl_rec_raw'] = quarterly['expenditure_rece_count'] / quarterly['denominator_count']
quarterly['Debt_FPU_raw'] = quarterly['debt_count'] / quarterly['denominator_count']
quarterly['Debt_FPU_excl_rec_raw'] = quarterly['debt_rece_count'] / quarterly['denominator_count']
quarterly['Other_FPU_raw'] = quarterly['other_count'] / quarterly['denominator_count']
quarterly['Other_FPU_excl_rec_raw'] = quarterly['other_rece_count'] / quarterly['denominator_count']

# Guard against division by zero (denominator_count == 0).
_quarterly_raw_cols = [
    'FPU_base_raw', 'FPU_excl_rec_raw',
    'Tax_FPU_raw', 'Tax_FPU_excl_rec_raw',
    'Expenditure_FPU_raw', 'Expenditure_FPU_excl_rec_raw',
    'Debt_FPU_raw', 'Debt_FPU_excl_rec_raw',
    'Other_FPU_raw', 'Other_FPU_excl_rec_raw',
]
quarterly[_quarterly_raw_cols] = quarterly[_quarterly_raw_cols].replace(
    [float('inf'), -float('inf')], pd.NA
)


def _z100(s):
    sd = s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(pd.NA, index=s.index, dtype='Float64')
    return (s - s.mean()) / sd + 100

for raw, transformed in {
    'FPU_base_raw': 'FPU_base_transform',
    'FPU_excl_rec_raw': 'FPU_excl_rec_transform',
    'Tax_FPU_raw': 'Tax_FPU_transform',
    'Tax_FPU_excl_rec_raw': 'Tax_FPU_excl_rec_transform',
    'Expenditure_FPU_raw': 'Expenditure_FPU_transform',
    'Expenditure_FPU_excl_rec_raw': 'Expenditure_FPU_excl_rec_transform',
    'Debt_FPU_raw': 'Debt_FPU_transform',
    'Debt_FPU_excl_rec_raw': 'Debt_FPU_excl_rec_transform',
    'Other_FPU_raw': 'Other_FPU_transform',
    'Other_FPU_excl_rec_raw': 'Other_FPU_excl_rec_transform',
}.items():
    quarterly[transformed] = quarterly.groupby('iso3')[raw].transform(_z100)

quarterly.rename(columns={'publication_datetime': 'quarter_end'}, inplace=True)
quarterly['quarter'] = quarterly['quarter_end'].dt.to_period('Q')

In [ ]:
# 5) Inspect
print(quarterly.head())

### Monthly

In [ ]:
# Monthly raw indices using a common denominator.
# IMPORTANT: aggregate numerator_base_count comes from the OR query and is NOT
# tax_count + expenditure_count + debt_count because subcategory overlap is allowed.
final_df['FPU_base_raw'] = final_df['numerator_base_count'] / final_df['denominator_count']
final_df['FPU_excl_rec_raw'] = final_df['numerator_rece_count'] / final_df['denominator_count']
final_df['Tax_FPU_raw'] = final_df['tax_count'] / final_df['denominator_count']
final_df['Tax_FPU_excl_rec_raw'] = final_df['tax_rece_count'] / final_df['denominator_count']
final_df['Expenditure_FPU_raw'] = final_df['expenditure_count'] / final_df['denominator_count']
final_df['Expenditure_FPU_excl_rec_raw'] = final_df['expenditure_rece_count'] / final_df['denominator_count']
final_df['Debt_FPU_raw'] = final_df['debt_count'] / final_df['denominator_count']
final_df['Debt_FPU_excl_rec_raw'] = final_df['debt_rece_count'] / final_df['denominator_count']
final_df['Other_FPU_raw'] = final_df['other_count'] / final_df['denominator_count']
final_df['Other_FPU_excl_rec_raw'] = final_df['other_rece_count'] / final_df['denominator_count']

# Guard against division by zero (denominator_count == 0).
_monthly_raw_cols = [
    'FPU_base_raw', 'FPU_excl_rec_raw',
    'Tax_FPU_raw', 'Tax_FPU_excl_rec_raw',
    'Expenditure_FPU_raw', 'Expenditure_FPU_excl_rec_raw',
    'Debt_FPU_raw', 'Debt_FPU_excl_rec_raw',
    'Other_FPU_raw', 'Other_FPU_excl_rec_raw',
]
final_df[_monthly_raw_cols] = final_df[_monthly_raw_cols].replace(
    [float('inf'), -float('inf')], pd.NA
)


def _normalize_monthly(s):
    sd = s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(pd.NA, index=s.index, dtype='Float64')
    return (s - s.mean()) / sd + 100

for raw, transformed in {
    'FPU_base_raw': 'FPU_base_transform',
    'FPU_excl_rec_raw': 'FPU_excl_rec_transform',
    'Tax_FPU_raw': 'Tax_FPU_transform',
    'Tax_FPU_excl_rec_raw': 'Tax_FPU_excl_rec_transform',
    'Expenditure_FPU_raw': 'Expenditure_FPU_transform',
    'Expenditure_FPU_excl_rec_raw': 'Expenditure_FPU_excl_rec_transform',
    'Debt_FPU_raw': 'Debt_FPU_transform',
    'Debt_FPU_excl_rec_raw': 'Debt_FPU_excl_rec_transform',
    'Other_FPU_raw': 'Other_FPU_transform',
    'Other_FPU_excl_rec_raw': 'Other_FPU_excl_rec_transform',
}.items():
    final_df[transformed] = final_df.groupby('iso3')[raw].transform(_normalize_monthly)

print("Monthly category/aggregate count columns:", [
    c for c in [
        'numerator_base_count', 'numerator_rece_count',
        'tax_count', 'tax_rece_count',
        'expenditure_count', 'expenditure_rece_count',
        'debt_count', 'debt_rece_count',
        'other_count', 'other_rece_count',
    ]
    if c in final_df.columns
])

In [ ]:
# ============================================================
# MASTER-DATASET QC FIELDS
# ============================================================
# Category counts are intentionally non-mutually-exclusive.
# Therefore, their sum can exceed the aggregate OR numerator.

final_df['subcategory_sum'] = final_df[
    ['tax_count', 'expenditure_count', 'debt_count']
].sum(axis=1)

final_df['subcategory_sum_minus_aggregate'] = (
    final_df['subcategory_sum'] - final_df['numerator_base_count']
)

# Logical checks: each component is a subset of the aggregate OR query.
final_df['qc_tax_le_aggregate'] = (
    final_df['tax_count'] <= final_df['numerator_base_count']
)
final_df['qc_expenditure_le_aggregate'] = (
    final_df['expenditure_count'] <= final_df['numerator_base_count']
)
final_df['qc_debt_le_aggregate'] = (
    final_df['debt_count'] <= final_df['numerator_base_count']
)
final_df['qc_other_le_aggregate'] = (
    final_df['other_count'] <= final_df['numerator_base_count']
)
final_df['qc_denominator_positive'] = final_df['denominator_count'] > 0

final_df['subcategory_rece_sum'] = final_df[
    ['tax_rece_count', 'expenditure_rece_count', 'debt_rece_count']
].sum(axis=1)
final_df['subcategory_rece_sum_minus_aggregate'] = (
    final_df['subcategory_rece_sum'] - final_df['numerator_rece_count']
)

# A recession-excluded count is a subset of BOTH its own plain count and the
# recession-excluded aggregate.
final_df['qc_rece_le_base'] = final_df['numerator_rece_count'] <= final_df['numerator_base_count']
for _c in ['tax', 'expenditure', 'debt', 'other']:
    final_df[f'qc_{_c}_rece_le_{_c}'] = (
        final_df[f'{_c}_rece_count'] <= final_df[f'{_c}_count']
    )
    final_df[f'qc_{_c}_rece_le_aggregate_rece'] = (
        final_df[f'{_c}_rece_count'] <= final_df['numerator_rece_count']
    )

qc = final_df[[
    'iso3', 'publication_datetime',
    'numerator_base_count', 'numerator_rece_count',
    'tax_count', 'tax_rece_count',
    'expenditure_count', 'expenditure_rece_count',
    'debt_count', 'debt_rece_count',
    'other_count', 'other_rece_count',
    'subcategory_sum', 'subcategory_sum_minus_aggregate',
    'subcategory_rece_sum', 'subcategory_rece_sum_minus_aggregate',
    'qc_tax_le_aggregate', 'qc_expenditure_le_aggregate',
    'qc_debt_le_aggregate', 'qc_other_le_aggregate', 'qc_denominator_positive',
    'qc_rece_le_base',
    'qc_tax_rece_le_tax', 'qc_tax_rece_le_aggregate_rece',
    'qc_expenditure_rece_le_expenditure', 'qc_expenditure_rece_le_aggregate_rece',
    'qc_debt_rece_le_debt', 'qc_debt_rece_le_aggregate_rece',
    'qc_other_rece_le_other', 'qc_other_rece_le_aggregate_rece',
]].copy()

print(qc['subcategory_sum_minus_aggregate'].describe())
print(
    'Rows with category overlap:',
    (qc['subcategory_sum_minus_aggregate'] > 0).sum()
)

_qc_bool_cols = [
    'qc_tax_le_aggregate', 'qc_expenditure_le_aggregate',
    'qc_debt_le_aggregate', 'qc_other_le_aggregate', 'qc_denominator_positive',
    'qc_rece_le_base',
    'qc_tax_rece_le_tax', 'qc_tax_rece_le_aggregate_rece',
    'qc_expenditure_rece_le_expenditure', 'qc_expenditure_rece_le_aggregate_rece',
    'qc_debt_rece_le_debt', 'qc_debt_rece_le_aggregate_rece',
    'qc_other_rece_le_other', 'qc_other_rece_le_aggregate_rece',
]
for _col in _qc_bool_cols:
    print(f"{_col}: {(~qc[_col]).sum()} False")

In [ ]:
# ============================================================
# EXPORT RAW MASTER INPUT FOR CLEANING NOTEBOOK
# ============================================================
# The monthly and quarterly sheets retain counts, raw ratios, normalized
# diagnostic indices, and overlap fields. 02_Clean_GFPU_MasterDataset.ipynb
# will apply the coverage filter and create the publication-ready master file.

monthly_master_cols = [
    'iso3', 'publication_datetime',
    'denominator_count',
    'numerator_base_count', 'numerator_rece_count',
    'tax_count', 'tax_rece_count',
    'expenditure_count', 'expenditure_rece_count',
    'debt_count', 'debt_rece_count',
    'other_count', 'other_rece_count',
    'subcategory_sum', 'subcategory_sum_minus_aggregate',
    'subcategory_rece_sum', 'subcategory_rece_sum_minus_aggregate',
    'FPU_base_raw', 'FPU_excl_rec_raw',
    'Tax_FPU_raw', 'Tax_FPU_excl_rec_raw',
    'Expenditure_FPU_raw', 'Expenditure_FPU_excl_rec_raw',
    'Debt_FPU_raw', 'Debt_FPU_excl_rec_raw',
    'Other_FPU_raw', 'Other_FPU_excl_rec_raw',
    'FPU_base_transform', 'FPU_excl_rec_transform',
    'Tax_FPU_transform', 'Tax_FPU_excl_rec_transform',
    'Expenditure_FPU_transform', 'Expenditure_FPU_excl_rec_transform',
    'Debt_FPU_transform', 'Debt_FPU_excl_rec_transform',
    'Other_FPU_transform', 'Other_FPU_excl_rec_transform',
]
monthly_master = final_df[[c for c in monthly_master_cols if c in final_df.columns]].copy()

# Add the same overlap diagnostics to quarterly output.
quarterly['subcategory_sum'] = quarterly[
    ['tax_count', 'expenditure_count', 'debt_count']
].sum(axis=1)
quarterly['subcategory_sum_minus_aggregate'] = (
    quarterly['subcategory_sum'] - quarterly['numerator_base_count']
)
quarterly['subcategory_rece_sum'] = quarterly[
    ['tax_rece_count', 'expenditure_rece_count', 'debt_rece_count']
].sum(axis=1)
quarterly['subcategory_rece_sum_minus_aggregate'] = (
    quarterly['subcategory_rece_sum'] - quarterly['numerator_rece_count']
)

quarterly_master_cols = [
    'iso3', 'quarter_end', 'quarter',
    'denominator_count',
    'numerator_base_count', 'numerator_rece_count',
    'tax_count', 'tax_rece_count',
    'expenditure_count', 'expenditure_rece_count',
    'debt_count', 'debt_rece_count',
    'other_count', 'other_rece_count',
    'subcategory_sum', 'subcategory_sum_minus_aggregate',
    'subcategory_rece_sum', 'subcategory_rece_sum_minus_aggregate',
    'FPU_base_raw', 'FPU_excl_rec_raw',
    'Tax_FPU_raw', 'Tax_FPU_excl_rec_raw',
    'Expenditure_FPU_raw', 'Expenditure_FPU_excl_rec_raw',
    'Debt_FPU_raw', 'Debt_FPU_excl_rec_raw',
    'Other_FPU_raw', 'Other_FPU_excl_rec_raw',
    'FPU_base_transform', 'FPU_excl_rec_transform',
    'Tax_FPU_transform', 'Tax_FPU_excl_rec_transform',
    'Expenditure_FPU_transform', 'Expenditure_FPU_excl_rec_transform',
    'Debt_FPU_transform', 'Debt_FPU_excl_rec_transform',
    'Other_FPU_transform', 'Other_FPU_excl_rec_transform',
]
quarterly_master = quarterly[[c for c in quarterly_master_cols if c in quarterly.columns]].copy()

output_file = os.path.join(cwd, 'outputs', 'FPU_Factiva_raw.xlsx')
os.makedirs(os.path.dirname(output_file), exist_ok=True)

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    monthly_master.to_excel(writer, index=False, sheet_name='monthly')
    quarterly_master.to_excel(writer, index=False, sheet_name='quarterly')
    qc.to_excel(writer, index=False, sheet_name='qc')
    final_df_raw.to_excel(writer, index=False, sheet_name='all_periods')

print(f'STAGE 3 FINAL INTEGRATED EXCEL SAVED: {output_file}')
print('Monthly master columns:')
print(monthly_master.columns.tolist())

In [ ]:
# Generate the filename with the current date for archive
current_datetime = datetime.now().strftime("%m%d%Y_%H%M")
archive_dir = f"{cwd}/Archives"

# Create Archives directory if it doesn't exist
os.makedirs(archive_dir, exist_ok=True)

archive_filename = f"{archive_dir}/FPU_Factiva_{current_datetime}.xlsx"

In [ ]:
# Save an archive copy using the same master structure
with pd.ExcelWriter(archive_filename, engine='openpyxl') as writer:
    monthly_master.to_excel(writer, index=False, sheet_name='monthly')
    quarterly_master.to_excel(writer, index=False, sheet_name='quarterly')
    qc.to_excel(writer, index=False, sheet_name='qc')

print(f'Export complete: {archive_filename}')


## Line charts

In [ ]:
country_iso3 = 'BEL'

In [ ]:
# Plot for the selected country (country_iso3)
df_country = final_df[final_df['iso3'] == country_iso3].sort_values('publication_datetime')

plt.figure()
plt.plot(df_country['publication_datetime'], df_country['FPU_base_transform'])
plt.xlabel('Publication Date')
plt.ylabel('FPU Index')
plt.title(f'Fiscal Policy Uncertainty for {country_iso3}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot for the selected country (country_iso3)
df_country = final_df[final_df['iso3'] == country_iso3].sort_values('publication_datetime')

plt.figure()
plt.plot(df_country['publication_datetime'], df_country['FPU_excl_rec_transform'])
plt.xlabel('Publication Date')
plt.ylabel('FPU Index')
plt.title(f'Fiscal Policy Uncertainty (Excluding Recessions Related Words) for {country_iso3}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot for the selected country (country_iso3)
df_country = final_df[final_df['iso3'] == country_iso3].sort_values('publication_datetime')

plt.figure()
plt.plot(df_country['publication_datetime'], df_country['Tax_FPU_transform'])
plt.xlabel('Publication Date')
plt.ylabel('Tax FPU Index')
plt.title(f'Tax Policy Uncertainty for {country_iso3}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot for the selected country (country_iso3)
df_country = final_df[final_df['iso3'] == country_iso3].sort_values('publication_datetime')

plt.figure()
plt.plot(df_country['publication_datetime'], df_country['Tax_FPU_excl_rec_transform'])
plt.xlabel('Publication Date')
plt.ylabel('Tax FPU Index')
plt.title(f'Tax Policy Uncertainty (Excluding Recessions Related Words) for {country_iso3}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot for the selected country (country_iso3)
df_country = final_df[final_df['iso3'] == country_iso3].sort_values('publication_datetime')

plt.figure()
plt.plot(df_country['publication_datetime'], df_country['Expenditure_FPU_transform'])
plt.xlabel('Publication Date')
plt.ylabel('Expenditure FPU Index')
plt.title(f'Expenditure Policy Uncertainty for {country_iso3}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot for the selected country (country_iso3)
df_country = final_df[final_df['iso3'] == country_iso3].sort_values('publication_datetime')

plt.figure()
plt.plot(df_country['publication_datetime'], df_country['Expenditure_FPU_excl_rec_transform'])
plt.xlabel('Publication Date')
plt.ylabel('Expenditure FPU Index')
plt.title(f'Expenditure Policy Uncertainty (Excluding Recessions Related Words) for {country_iso3}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot for the selected country (country_iso3)
df_country = final_df[final_df['iso3'] == country_iso3].sort_values('publication_datetime')

plt.figure()
plt.plot(df_country['publication_datetime'], df_country['Debt_FPU_transform'])
plt.xlabel('Publication Date')
plt.ylabel('Debt FPU Index')
plt.title(f'Debt Policy Uncertainty for {country_iso3}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot for the selected country (country_iso3)
df_country = final_df[final_df['iso3'] == country_iso3].sort_values('publication_datetime')

plt.figure()
plt.plot(df_country['publication_datetime'], df_country['Debt_FPU_excl_rec_transform'])
plt.xlabel('Publication Date')
plt.ylabel('Debt FPU Index')
plt.title(f'Debt Policy Uncertainty (Excluding Recessions Related Words) for {country_iso3}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()